In [3]:
# ============================================================
# 🛰️ Earth Risk Simulator (Baseline Version)
# Part of Interplanetix Global Satellite Risk Tool
# ============================================================

# --- 1. Imports ---
import os
import sys   # 🔧 Needed for modifying Python import paths
import pandas as pd
import plotly.graph_objects as go
from datetime import datetime, timezone
from skyfield.api import load, EarthSatellite


# ✅ Get project root relative to where notebook is running
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))

# ✅ Add to sys.path if not already there
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("🔧 Project root added to path:", PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, "data")

tle_path = os.path.join(DATA_DIR, "tle_data.txt")


# --- 2. Paths ---
DATA_DIR = "../data"
VIS_DIR = "../visualizations/risk_maps"
DOCS_HTML_PATH = "../docs/satellite_risk_map_sample.html"

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

# --- 3. Load TLE file ---
# You can replace with your latest TLE file
TLE_FILE = os.path.join(DATA_DIR, "tle_data_20250626.txt")

with open(TLE_FILE, "r") as f:
    lines = f.readlines()

sats = []
for i in range(0, len(lines), 3):
    try:
        name = lines[i].strip()
        l1 = lines[i+1].strip()
        l2 = lines[i+2].strip()
        sat = EarthSatellite(l1, l2, name, load.timescale())
        sats.append(sat)
    except Exception as e:
        print(f"⚠️ Skipping {name} due to error: {e}")

print(f"✅ Loaded {len(sats)} satellites")

# --- 4. Propagate positions ---
ts = load.timescale()
t = ts.now()

records = []
for sat in sats:
    try:
        geo = sat.at(t).subpoint()
        lat, lon, alt = geo.latitude.degrees, geo.longitude.degrees, geo.elevation.km
        incl = sat.model.inclo

        # Classify risk by inclination (very simple rule for demo)
        if incl < 30:
            risk = "Low"
        elif incl < 70:
            risk = "Medium"
        else:
            risk = "High"

        records.append({
            "name": sat.name,
            "latitude": lat,
            "longitude": lon,
            "altitude_km": alt,
            "inclination_deg": incl,
            "risk_level": risk
        })
    except Exception as e:
        print(f"⚠️ Could not compute {sat.name}: {e}")

df = pd.DataFrame(records)
print("📊 Sample data:")
print(df.head())

# --- 5. Save CSV ---
today = datetime.now(timezone.utc).strftime("%Y-%m-%d")
csv_file = os.path.join(VIS_DIR, f"{today}_risk_data.csv")
df.to_csv(csv_file, index=False)
print(f"✅ Saved risk data → {csv_file}")

# --- 6. Plot interactive map ---
color_map = {"Low": "green", "Medium": "orange", "High": "red"}
df["color"] = df["risk_level"].map(color_map)

fig = go.Figure(go.Scattergeo(
    lon = df["longitude"],
    lat = df["latitude"],
    text = df["name"] + "<br>Risk: " + df["risk_level"],
    mode = "markers",
    marker = dict(size=6, color=df["color"], line=dict(width=0.5, color="black"), opacity=0.8),
))

fig.update_layout(
    title="🛰️ Global Satellite Risk Map (Earth Only)",
    geo=dict(showland=True, landcolor="rgb(243,243,243)", countrycolor="rgb(204,204,204)")
)

html_file = os.path.join(VIS_DIR, f"{today}_satellite_risk_map.html")
fig.write_html(html_file)
print(f"✅ Saved HTML map → {html_file}")

# --- 7. Copy to docs for website embedding ---
from shutil import copyfile
copyfile(html_file, DOCS_HTML_PATH)
print(f"✅ Copied latest map → {DOCS_HTML_PATH}")

fig.show()


🔧 Project root added to path: c:\Users\DELL\OneDrive\Desktop\Interplanix\Git\interplanix\global-satellite-risk-tool


FileNotFoundError: [Errno 2] No such file or directory: '../data\\tle_data_20250626.txt'